<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/HAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset, DataLoader

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = '128'

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

!pip install torch-sparse torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

!pip install torch-geometric
import torch_geometric

#import torch_sparse
#import torch_scatter

#from scipy.ndimage import maximum_filter
#from scipy.spatial._qhull import ConvexHull

import sys

import matplotlib.pyplot as plt

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.2 MB/s eta 0:00:00


In [ ]:
class MetaPathPassing(nn.Module):

  def __init__(self, in_channels, out_channels, dropout = 0.2):

    super().__init__()
    self.GAT = torch_geometric.nn.conv.GATv2Conv(in_channels, out_channels, heads = 4, dropout = 0.2)


In [19]:
class HAN(nn.Module):

  def __init__(self, metapaths, in_channels, hidden_channels, heads = 4, dropout = 0.2):

    super().__init__()
    self.metapaths = list(metapaths.keys())
    self.MetaPathNetworks = nn.ModuleDict({path: torch_geometric.nn.conv.GATv2Conv(in_channels, hidden_channels[0], heads = heads, dropout = dropout)
                                           for path, (d_src, d_dst) in metapaths.items()})
    self.SemanticNetwork = nn.Sequential(nn.Linear(hidden_channels[0],hidden_channels[1]), nn.Tanh(), nn.Linear(hidden_channels[1], 1, bias = False))

  def forward(self, x: dict, edge_indices: dict): #both are dictionaries, keys corresponding to each metapath

    Z_m = [F.elu(self.MetaPathNetworks[mp](x[mp], edge_indices[mp])) for mp in self.metapaths] #each [N, D]
    Z_m = torch.stack(Z_m, dim = 1) #(N, M, D)

    w_m = self.SematicNetwork(Z_m).mean(dim=0) #semanticnet maps all nodes individaully, then take mean for semantic level embedding
    beta_m = F.softmax(w_m, dim = 0)
    out = (beta_m.unsqueeze(0) * Z_m).sum(dim =1) #sum over all metapaths

    return out, beta_m.squeeze()



In [20]:
class HAN_Stack(nn.Module):

  def __init__(self, d_lidar, d_camera, hidden_channels=[64,64], num_layers =2, heads = 4, dropout = 0.2): #dims of lidar and camera feature nodes, currently only works with 2 layers
    super().__init__()
    self.n_layers = num_layers

    self.lidar_layers = nn.ModuleList()
    self.camera_layers = nn.ModuleList()
    self.layer_out = hidden_channels * heads

    for i in range(num_layers):
      dl = d_lidar if i == 0 else self.layer_out
      dc = d_camera if i == 0 else self.layer_out

      self.lidar_layers.append(HAN({"LL": (dl, dl), "LCL": (dl,dl)}, dl, hidden_channels, heads = heads, dropout = dropout)) #dicts contain src and dst feature tensors
      self.camera_layers.append(HAN({"CC": (dc, dc), "CLC": (dc,dc)}, dc, hidden_channels, heads = heads, dropout = dropout))

  def forward(self, x_lidar, x_camera, edge_ll, edge_cc, edge_lcl, edge_clc):

    for i in range(self.n_layers):
      new_lidar, beta_l = self.lidar_layers[i]({"LL": x_lidar, "LCL": x_lidar}, {"LL": edge_ll, "LCL": edge_lcl})
      new_camera, beta_c = self.lidar_layers[i]({"CC": x_camera, "CLC": x_camera}, {"CC": edge_cc, "CLC": edge_clc})

      x_lidar, x_camera = new_lidar, new_camera

    return new_lidar, new_camera, beta_l, beta_c

In [9]:
def build_metapath(src, n_src, dst, n_dst, top_k =16): #src and dst are edge indices
  device = src.device

  adj_ab = torch.sparse_coo_tensor( #stores only nonzero elements(values), and their indices
        src, torch.ones(src.size(1), device=device),
        size=(n_src, n_dst)
    ).coalesce()
  adj_ba = torch.sparse_coo_tensor(
        n_dst, torch.ones(n_dst.size(1), device=device),
        size=(n_dst, n_src)
    ).coalesce()

  adj_aa = torch.sparse.mm(adj_ab, adj_ba).coalesce() #mm = matmul, coalesce combines duplicate indices and sorts indices
  idx = adj_aa.indices()
  val = adj_aa.values()

  #top-k sorting, look into later

  keep = (idx[0] != idx[1])
  idx, val = idx[:, keep], val[keep]

  order = torch.argsort(val, descending = True)
  idx, val = idx[:, order], val[order]

  a = idx[0]
  perm = torch.argsort(a, stable = True)
  idx, val, a = idx[:, perm], val[perm], a[perm]

  counts = torch.zeros((n_src), dtype=torch.long, device=device)
  keep_mask = torch.zeros(a.size(0), dtype=torch.bool, device=device)

  _, first_pos = torch.unique_consecutive(a, return_counts=True)
  rank = torch.cat([torch.arange(c, device=device) for c in first_pos])
  keep_mask = rank < top_k

  return idx[:, keep_mask]
